In [15]:
# ============================================================
# BAGIAN B - DATA INGESTION & INTEGRATION
# ============================================================

import pandas as pd
import sqlite3
import json
from pathlib import Path


# ============================================================
# 0. PENGATURAN FILE
# ============================================================

BASE_DIR = Path(".")

FILE_CSV = BASE_DIR / "transaksi_penjualan.csv"
FILE_DB = BASE_DIR / "crm_membership.sqlite"
FILE_JSON = BASE_DIR / "mock_logistics_api.json"
FILE_OUTPUT = BASE_DIR / "data_master_analisis.csv"


# ============================================================
# 1. CEK FILE TRANSAKSI
# ============================================================

if not FILE_CSV.exists():
    raise FileNotFoundError(
        "File transaksi_penjualan.csv tidak ditemukan. "
        "Silakan upload file tersebut ke Files di Google Colab."
    )


# ============================================================
# 2. EKSTRAKSI DATA TRANSAKSI DARI CSV
# ============================================================

df_transaksi = pd.read_csv(
    FILE_CSV,
    dtype={
        "order_id": str,
        "customer_id": str
    }
)

# Membersihkan spasi pada ID
df_transaksi["order_id"] = (
    df_transaksi["order_id"]
    .astype("string")
    .str.strip()
)

df_transaksi["customer_id"] = (
    df_transaksi["customer_id"]
    .astype("string")
    .str.strip()
)

# Mengubah order_date menjadi datetime
df_transaksi["order_date"] = pd.to_datetime(
    df_transaksi["order_date"],
    errors="coerce"
)

print("=" * 60)
print("1. DATA TRANSAKSI")
print("=" * 60)

print(df_transaksi)

print("\nJumlah transaksi:", len(df_transaksi))
print("Kolom transaksi:")
print(df_transaksi.columns.tolist())


# ============================================================
# 3. MEMBUAT DATABASE CRM
# ============================================================

# Mengambil customer_id unik dari data transaksi
customer_ids = (
    df_transaksi["customer_id"]
    .dropna()
    .unique()
    .tolist()
)

# Data contoh untuk CRM
nama_depan = [
    "Andi", "Budi", "Citra", "Dina", "Eko",
    "Fajar", "Gita", "Hana", "Indra", "Joko",
    "Karin", "Lina", "Maya", "Nanda", "Oki",
    "Putri", "Raka", "Sinta", "Tio", "Vina"
]

gender_list = ["L", "P"]

membership_list = [
    "Bronze",
    "Silver",
    "Gold",
    "Platinum"
]

# Membuat data customer
data_customer = []

for i, customer_id in enumerate(customer_ids):

    customer_name = nama_depan[i % len(nama_depan)]

    gender = gender_list[i % len(gender_list)]

    membership_level = membership_list[
        i % len(membership_list)
    ]

    # Semua customer dibuat aktif
    is_active = 1

    data_customer.append(
        (
            str(customer_id),
            customer_name,
            gender,
            membership_level,
            is_active
        )
    )


# ============================================================
# 4. MEMBUAT TABEL CUSTOMERS DI SQLITE
# ============================================================

conn = sqlite3.connect(FILE_DB)

# Hapus tabel lama agar tidak terjadi konflik data
conn.execute("DROP TABLE IF EXISTS customers")

# Membuat tabel baru
conn.execute(
    """
    CREATE TABLE customers (
        customer_id TEXT PRIMARY KEY,
        customer_name TEXT,
        gender TEXT,
        membership_level TEXT,
        is_active INTEGER
    )
    """
)

# Memasukkan data customer
conn.executemany(
    """
    INSERT INTO customers
    (
        customer_id,
        customer_name,
        gender,
        membership_level,
        is_active
    )
    VALUES (?, ?, ?, ?, ?)
    """,
    data_customer
)

conn.commit()

print("\n" + "=" * 60)
print("2. DATABASE CRM")
print("=" * 60)

print("Database berhasil dibuat:", FILE_DB)


# ============================================================
# 5. QUERY DATA CUSTOMER AKTIF
# ============================================================

query = """
SELECT
    customer_id,
    customer_name,
    gender,
    membership_level,
    is_active
FROM customers
WHERE is_active = 1
"""

df_customer = pd.read_sql_query(
    query,
    conn
)

conn.close()

print("\n=== DATA CUSTOMER AKTIF ===")
print(df_customer)

print("\nJumlah customer aktif:", len(df_customer))


# ============================================================
# 6. MEMBUAT MOCK LOGISTICS API
# ============================================================

couriers = [
    "JNE",
    "J&T Express",
    "SiCepat",
    "AnterAja"
]

statuses = [
    "Processing",
    "Shipped",
    "Delivered"
]

logistics_data = []

# Mengambil order_id unik
order_ids = (
    df_transaksi["order_id"]
    .dropna()
    .unique()
    .tolist()
)

for i, order_id in enumerate(order_ids):

    logistics_data.append(
        {
            "order_id": str(order_id),

            "shipment": {
                "courier": couriers[
                    i % len(couriers)
                ],

                "status": statuses[
                    i % len(statuses)
                ],

                "tracking_number":
                    f"TRK{str(i + 1).zfill(6)}"
            }
        }
    )


# Struktur JSON seperti REST API
api_response = {
    "data": logistics_data
}


# Simpan JSON
with open(
    FILE_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        api_response,
        f,
        indent=4,
        ensure_ascii=False
    )


print("\n" + "=" * 60)
print("3. MOCK LOGISTICS API")
print("=" * 60)

print("File JSON berhasil dibuat:", FILE_JSON)


# ============================================================
# 7. MEMBACA DAN MENORMALISASI DATA JSON
# ============================================================

with open(
    FILE_JSON,
    "r",
    encoding="utf-8"
) as f:

    api_response = json.load(f)


df_logistik = pd.json_normalize(
    api_response["data"]
)


# Mengubah nama kolom agar lebih mudah dibaca
df_logistik = df_logistik.rename(
    columns={
        "shipment.courier": "courier",
        "shipment.status": "shipment_status",
        "shipment.tracking_number": "tracking_number"
    }
)


# Memastikan order_id bertipe string
df_logistik["order_id"] = (
    df_logistik["order_id"]
    .astype("string")
    .str.strip()
)


print("\n=== DATA LOGISTIK ===")
print(df_logistik)

print("\nKolom logistik:")
print(df_logistik.columns.tolist())


# ============================================================
# 8. MERGE TRANSAKSI + CUSTOMER
# ============================================================

df_master_analisis = df_transaksi.merge(
    df_customer,
    on="customer_id",
    how="left"
)


print("\n" + "=" * 60)
print("4. HASIL MERGE TRANSAKSI + CUSTOMER")
print("=" * 60)

print(df_master_analisis)


# ============================================================
# 9. MERGE + DATA LOGISTIK
# ============================================================

df_master_analisis = df_master_analisis.merge(
    df_logistik,
    on="order_id",
    how="left"
)


# ============================================================
# 10. HASIL AKHIR DATA MASTER
# ============================================================

print("\n" + "=" * 60)
print("5. DATA MASTER ANALISIS")
print("=" * 60)

print(df_master_analisis)


print("\nJumlah baris:", len(df_master_analisis))
print(
    "Jumlah kolom:",
    len(df_master_analisis.columns)
)


# ============================================================
# 11. CEK DATA YANG HILANG
# ============================================================

print("\n" + "=" * 60)
print("6. CEK MISSING VALUE")
print("=" * 60)

print(
    df_master_analisis.isnull().sum()
)


# ============================================================
# 12. SIMPAN HASIL AKHIR
# ============================================================

df_master_analisis.to_csv(
    FILE_OUTPUT,
    index=False
)

print("\n" + "=" * 60)
print("7. SELESAI")
print("=" * 60)

print(
    "Data master berhasil disimpan sebagai:"
)
print(FILE_OUTPUT)

print("\nFile yang tersedia:")
print("- transaksi_penjualan.csv")
print("- crm_membership.sqlite")
print("- mock_logistics_api.json")
print("- data_master_analisis.csv")

1. DATA TRANSAKSI
    order_id customer_id order_date product_category  quantity  unit_price  \
0   ORD-1001    CUST-001 2026-09-01       Elektronik         2      150000   
1   ORD-1002    CUST-002 2026-09-01          Fashion         1      275000   
2   ORD-1003    CUST-003 2026-09-02        Groceries         5       45000   
3   ORD-1004    CUST-004 2026-09-02             Home         1      425000   
4   ORD-1005    CUST-005 2026-09-03           Beauty         3       85000   
5   ORD-1006    CUST-006 2026-09-03       Elektronik         1      850000   
6   ORD-1007    CUST-007 2026-09-04          Fashion         2      175000   
7   ORD-1008    CUST-008 2026-09-04        Groceries         4       35000   
8   ORD-1009    CUST-009 2026-09-05             Home         2      320000   
9   ORD-1010    CUST-001 2026-09-05           Beauty         2       95000   
10  ORD-1011    CUST-002 2026-09-06       Elektronik         1      625000   
11  ORD-1012    CUST-003 2026-09-06          F

In [16]:
# ============================================================
# BAGIAN C - METADATA, KAMUS DATA & VALIDASI ATURAN BISNIS
# ============================================================

import pandas as pd


# ============================================================
# 1. OTOMASI KAMUS DATA (DATA DICTIONARY)
# ============================================================

def buat_kamus_data(df):
    """
    Fungsi untuk menghasilkan ringkasan metadata
    dari DataFrame.

    Informasi yang dihasilkan:
    - Nama kolom
    - Tipe data
    - Jumlah missing value
    - Persentase missing value
    - Jumlah nilai unik
    - Sampel nilai
    """

    hasil = []

    for kolom in df.columns:

        # Jumlah missing value
        jumlah_missing = df[kolom].isna().sum()

        # Persentase missing value
        persentase_missing = (
            jumlah_missing / len(df) * 100
            if len(df) > 0
            else 0
        )

        # Jumlah nilai unik
        jumlah_unik = df[kolom].nunique(
            dropna=True
        )

        # Mengambil maksimal 5 contoh nilai
        sampel_nilai = (
            df[kolom]
            .dropna()
            .astype(str)
            .unique()[:5]
        )

        # Mengubah sampel menjadi string
        sampel_nilai = ", ".join(sampel_nilai)

        hasil.append({
            "nama_kolom": kolom,
            "tipe_data": str(df[kolom].dtype),
            "jumlah_missing": jumlah_missing,
            "persentase_missing": round(
                persentase_missing, 2
            ),
            "jumlah_nilai_unik": jumlah_unik,
            "sampel_nilai": sampel_nilai
        })

    return pd.DataFrame(hasil)


# ============================================================
# 2. MEMBUAT KAMUS DATA DARI DATA MASTER
# ============================================================

kamus_data = buat_kamus_data(
    df_master_analisis
)


print("=" * 70)
print("KAMUS DATA / DATA DICTIONARY")
print("=" * 70)

print(kamus_data)


# ============================================================
# 3. MENYIMPAN KAMUS DATA KE CSV
# ============================================================

kamus_data.to_csv(
    "kamus_data_hasil.csv",
    index=False
)

print("\nKamus data berhasil disimpan sebagai:")
print("kamus_data_hasil.csv")


# ============================================================
# 4. VALIDASI KUALITAS & INTEGRITAS DATA
# ============================================================

print("\n" + "=" * 70)
print("VALIDASI KUALITAS & INTEGRITAS DATA")
print("=" * 70)


# ============================================================
# VALIDASI 1
# Tidak boleh ada duplikasi pada primary key
# Primary key yang digunakan: order_id
# ============================================================

jumlah_duplikat_order = (
    df_master_analisis["order_id"]
    .duplicated()
    .sum()
)

print(
    "Jumlah duplikasi order_id:",
    jumlah_duplikat_order
)

assert jumlah_duplikat_order == 0, (
    "VALIDASI GAGAL: Ditemukan duplikasi pada order_id!"
)

print(
    "✓ VALIDASI PRIMARY KEY: LULUS"
)


# ============================================================
# VALIDASI 2
# total_amount tidak boleh bernilai negatif
# ============================================================

jumlah_total_negatif = (
    df_master_analisis["total_amount"] < 0
).sum()

print(
    "Jumlah total_amount negatif:",
    jumlah_total_negatif
)

assert jumlah_total_negatif == 0, (
    "VALIDASI GAGAL: Ditemukan total_amount bernilai negatif!"
)

print(
    "✓ VALIDASI TOTAL_AMOUNT: LULUS"
)


# ============================================================
# VALIDASI 3
# order_date tidak boleh melebihi tanggal hari ini
# ============================================================

tanggal_hari_ini = pd.Timestamp.today().normalize()

jumlah_tanggal_masa_depan = (
    df_master_analisis["order_date"]
    > tanggal_hari_ini
).sum()

print(
    "Tanggal hari ini:",
    tanggal_hari_ini.date()
)

print(
    "Jumlah order dengan tanggal masa depan:",
    jumlah_tanggal_masa_depan
)

assert jumlah_tanggal_masa_depan == 0, (
    "VALIDASI GAGAL: Ditemukan order_date "
    "yang melebihi tanggal hari ini!"
)

print(
    "✓ VALIDASI ORDER_DATE: LULUS"
)


# ============================================================
# 5. RINGKASAN HASIL VALIDASI
# ============================================================

print("\n" + "=" * 70)
print("RINGKASAN VALIDASI")
print("=" * 70)

print("✓ Primary key order_id tidak duplikat")
print("✓ total_amount tidak bernilai negatif")
print("✓ order_date tidak melebihi tanggal hari ini")

print("\nSemua validasi berhasil!")

KAMUS DATA / DATA DICTIONARY
          nama_kolom       tipe_data  jumlah_missing  persentase_missing  \
0           order_id          string               0                 0.0   
1        customer_id          object               0                 0.0   
2         order_date  datetime64[ns]               0                 0.0   
3   product_category          object               0                 0.0   
4           quantity           int64               0                 0.0   
5         unit_price           int64               0                 0.0   
6       total_amount           int64               0                 0.0   
7        coupon_code          object              12                40.0   
8      customer_name          object               0                 0.0   
9             gender          object               0                 0.0   
10  membership_level          object               0                 0.0   
11         is_active           int64               0       